# ManiSkill #1495 — online GPU validation

Runs one frozen variant of the maintainer-requested PegInsertionSide Diffusion Policy comparison. Use a fresh Kaggle/Colab GPU session for each variant. Do not change the frozen training hyperparameters between baseline and fixed runs.


In [ ]:
import os, subprocess, pathlib
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
root = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path('/kaggle/working')
repo = root / 'ManiSkill-pr1495-validation'
if not repo.exists():
    subprocess.run(['git','clone','--branch','validation/maniskill1495-maintainer-request','https://github.com/lindicaphxag-tech/ManiSkill.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
print(repo)


In [ ]:
from getpass import getpass
os.environ['WANDB_API_KEY'] = getpass('W&B API key: ')
os.environ['WANDB_PROJECT'] = 'maniskill-pr1495-peg-dp'
# Choose exactly one per clean GPU session: 'baseline' or 'fixed'.
VARIANT = 'baseline'
SEED = 1
print('variant=', VARIANT, 'seed=', SEED)


In [ ]:
env = os.environ.copy()
env['VARIANT'] = VARIANT
env['SEED'] = str(SEED)
subprocess.run(['bash', str(repo/'validation/pr1495/run_peg_dp.sh')], cwd=repo, env=env, check=True)


In [ ]:
evidence_root = repo / '.validation-pr1495' / 'evidence'
for p in sorted(evidence_root.glob('*')):
    print(p)
    summary = p / 'summary.json'
    if summary.exists():
        print(summary.read_text())


After the paired seed-1 runs finish, preserve the two W&B run URLs plus each `manifest.json` and `summary.json`. Only then decide whether to run seeds 2 and 3. Null/negative outcomes are retained.
